# Kubeflow Pipeline: Submit PyTorch TrainJob

This notebook builds a **Kubeflow Pipeline (KFP)** that submits a Kubeflow Trainer `TrainJob` and waits for completion.

Key requirement implemented:
- `namespace` is a **pipeline input parameter** and is passed explicitly when creating the TrainJob.

In [ ]:
# !pip install -U kfp kubeflow

from kfp import Client, compiler, dsl

# User-provided values for compile/submit
TARGET_NAMESPACE = "kapil-test-namespace"  # <- set by user
KFP_HOST = ""  # optional; set if running outside cluster
EXPERIMENT_NAME = "feast-trainjob"
PIPELINE_PACKAGE_PATH = "/tmp/feast_trainjob_pipeline.yaml"

if not TARGET_NAMESPACE:
    raise ValueError("TARGET_NAMESPACE must be set")
print(f"Target namespace for TrainJob: {TARGET_NAMESPACE}")

In [ ]:
@dsl.component(
    base_image="python:3.11",
    packages_to_install=["kubeflow", "torch", "pandas", "scikit-learn", "pyarrow"],
)
def submit_trainjob_and_wait(
    namespace: str,
    runtime_name: str = "torch-distributed",
    num_nodes: int = 2,
    cpu_per_node: str = "2",
    memory_per_node: str = "4Gi",
    training_data_path: str = "/mnt/data/feast_training.parquet",
    metadata_path: str = "/mnt/data/feast_training_metadata.json",
    output_dir: str = "/mnt/models",
) -> str:
    def train_fraud_from_feast(
        num_epochs=10,
        batch_size=512,
        lr=1e-3,
        hidden_dim=64,
        training_data_path="/mnt/data/feast_training.parquet",
        metadata_path="/mnt/data/feast_training_metadata.json",
        output_dir="/mnt/models",
    ):
        import json
        import os
        import random

        import numpy as np
        import pandas as pd
        import torch
        import torch.distributed as dist
        from sklearn.metrics import roc_auc_score
        from torch import nn
        from torch.utils.data import DataLoader, Dataset, DistributedSampler

        random.seed(42)
        np.random.seed(42)
        torch.manual_seed(42)

        class TabularDataset(Dataset):
            def __init__(self, frame, feature_cols, label_col):
                self.x = torch.tensor(frame[feature_cols].values, dtype=torch.float32)
                self.y = torch.tensor(frame[label_col].values, dtype=torch.float32).unsqueeze(1)

            def __len__(self):
                return len(self.x)

            def __getitem__(self, idx):
                return self.x[idx], self.y[idx]

        class FraudMLP(nn.Module):
            def __init__(self, input_dim, hidden):
                super().__init__()
                self.net = nn.Sequential(
                    nn.Linear(input_dim, hidden),
                    nn.ReLU(),
                    nn.Dropout(0.2),
                    nn.Linear(hidden, hidden // 2),
                    nn.ReLU(),
                    nn.Linear(hidden // 2, 1),
                )

            def forward(self, x):
                return self.net(x)

        if not os.path.exists(training_data_path):
            raise FileNotFoundError(f"Missing dataset: {training_data_path}")

        with open(metadata_path, "r", encoding="utf-8") as f:
            metadata = json.load(f)

        feature_cols = metadata["feature_columns"]
        label_col = metadata["label_column"]

        df = pd.read_parquet(training_data_path)
        train_df = df[df["split"] == "train"].copy()
        val_df = df[df["split"] == "val"].copy()
        if train_df.empty or val_df.empty:
            raise RuntimeError("Train/val splits are empty")

        world_size = int(os.getenv("WORLD_SIZE", "1"))
        rank = int(os.getenv("RANK", "0"))
        local_rank = int(os.getenv("LOCAL_RANK", "0"))

        distributed = world_size > 1
        if distributed:
            backend = "nccl" if torch.cuda.is_available() else "gloo"
            dist.init_process_group(backend=backend)

        if torch.cuda.is_available():
            device = torch.device(f"cuda:{local_rank}")
            torch.cuda.set_device(device)
        else:
            device = torch.device("cpu")

        model = FraudMLP(input_dim=len(feature_cols), hidden=hidden_dim).to(device)
        if distributed:
            model = nn.parallel.DistributedDataParallel(
                model,
                device_ids=[local_rank] if torch.cuda.is_available() else None,
            )

        train_dataset = TabularDataset(train_df, feature_cols, label_col)
        val_dataset = TabularDataset(val_df, feature_cols, label_col)

        train_sampler = DistributedSampler(train_dataset) if distributed else None
        train_loader = DataLoader(
            train_dataset,
            batch_size=batch_size,
            sampler=train_sampler,
            shuffle=(train_sampler is None),
        )
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

        criterion = torch.nn.BCEWithLogitsLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)

        for epoch in range(num_epochs):
            model.train()
            if train_sampler is not None:
                train_sampler.set_epoch(epoch)

            running_loss = 0.0
            for xb, yb in train_loader:
                xb, yb = xb.to(device), yb.to(device)
                optimizer.zero_grad()
                logits = model(xb)
                loss = criterion(logits, yb)
                loss.backward()
                optimizer.step()
                running_loss += loss.item()

            model.eval()
            val_targets = []
            val_scores = []
            with torch.no_grad():
                for xb, yb in val_loader:
                    xb = xb.to(device)
                    logits = model(xb)
                    probs = torch.sigmoid(logits).cpu().numpy().reshape(-1)
                    val_scores.extend(probs.tolist())
                    val_targets.extend(yb.numpy().reshape(-1).tolist())

            if rank == 0 and len(set(int(v) for v in val_targets)) >= 2:
                val_auc = roc_auc_score(val_targets, val_scores)
                print(f"Epoch {epoch + 1}/{num_epochs} | loss={running_loss / max(len(train_loader), 1):.4f} | val_auc={val_auc:.4f}")

        if rank == 0:
            os.makedirs(output_dir, exist_ok=True)
            model_to_save = model.module if hasattr(model, "module") else model
            torch.save(
                {
                    "model_state_dict": model_to_save.state_dict(),
                    "feature_columns": feature_cols,
                    "label_column": label_col,
                    "hidden_dim": hidden_dim,
                },
                os.path.join(output_dir, "fraud_mlp_state_dict.pt"),
            )

        if distributed:
            dist.barrier()
            dist.destroy_process_group()

    from kubeflow.trainer import CustomTrainer, TrainerClient

    if not namespace:
        raise ValueError("namespace must be provided")

    train_kwargs = {}
    wait_kwargs = {}
    try:
        client = TrainerClient(namespace=namespace)
    except TypeError:
        client = TrainerClient()
        train_kwargs["namespace"] = namespace
        wait_kwargs["namespace"] = namespace

    job_name = client.train(
        trainer=CustomTrainer(
            func=train_fraud_from_feast,
            func_args={
                "num_epochs": 10,
                "batch_size": 512,
                "lr": 1e-3,
                "hidden_dim": 64,
                "training_data_path": training_data_path,
                "metadata_path": metadata_path,
                "output_dir": output_dir,
            },
            num_nodes=num_nodes,
            resources_per_node={"cpu": cpu_per_node, "memory": memory_per_node},
            packages_to_install=["torch", "pandas", "scikit-learn", "pyarrow"],
        ),
        runtime=runtime_name,
        **train_kwargs,
    )

    client.wait_for_job_status(name=job_name, status={"Running"}, timeout=600, **wait_kwargs)
    client.wait_for_job_status(name=job_name, timeout=60, **wait_kwargs)
    print(f"TrainJob completed: {job_name} in namespace {namespace}")
    return job_name

In [ ]:
@dsl.pipeline(name="feast-pytorch-trainjob-pipeline")
def feast_pytorch_trainjob_pipeline(
    namespace: str,
    runtime_name: str = "torch-distributed",
    num_nodes: int = 2,
    cpu_per_node: str = "2",
    memory_per_node: str = "4Gi",
    training_data_path: str = "/mnt/data/feast_training.parquet",
    metadata_path: str = "/mnt/data/feast_training_metadata.json",
    output_dir: str = "/mnt/models",
):
    submit_trainjob_and_wait(
        namespace=namespace,
        runtime_name=runtime_name,
        num_nodes=num_nodes,
        cpu_per_node=cpu_per_node,
        memory_per_node=memory_per_node,
        training_data_path=training_data_path,
        metadata_path=metadata_path,
        output_dir=output_dir,
    )

print("Pipeline defined")

In [ ]:
compiler.Compiler().compile(
    pipeline_func=feast_pytorch_trainjob_pipeline,
    package_path=PIPELINE_PACKAGE_PATH,
)
print(f"Compiled pipeline: {PIPELINE_PACKAGE_PATH}")

In [ ]:
# Optional: submit run to KFP
# If running in-cluster, Client() may work without host.
if KFP_HOST:
    kfp_client = Client(host=KFP_HOST)
else:
    kfp_client = Client()

run = kfp_client.create_run_from_pipeline_package(
    pipeline_file=PIPELINE_PACKAGE_PATH,
    experiment_name=EXPERIMENT_NAME,
    arguments={
        "namespace": TARGET_NAMESPACE,
        "runtime_name": "torch-distributed",
        "num_nodes": 2,
        "cpu_per_node": "2",
        "memory_per_node": "4Gi",
        "training_data_path": "/mnt/data/feast_training.parquet",
        "metadata_path": "/mnt/data/feast_training_metadata.json",
        "output_dir": "/mnt/models",
    },
)
print(f"Pipeline run submitted: {run.run_id}")